# Tutorial: AlphaPose Colab Inference

What this notebook teaches:
- Try the recommended AlphaPose setup path in Colab.
- Detect runtime setup failures and switch to fallback mode.
- Map AlphaPose-style COCO keypoints to canonical schema and export outputs.
- Persist benchmark status without fabricated metrics.


In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/sumeyye-agac/human-pose-estimation-experiments.git"
REPO_DIR = Path("/content/human-pose-estimation-experiments")

if "google.colab" in sys.modules:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    raise RuntimeError("Run this notebook from the repository root.")

if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

print(f"Using repo root: {repo_root}")


## Recommended path

AlphaPose setup can break due CUDA and dependency pinning. This notebook checks for an importable runtime first.


In [ ]:
import json

alphapose_ready = False
alphapose_error = None

try:
    import alphapose  # type: ignore

    alphapose_ready = True
except Exception as exc:
    alphapose_error = str(exc)

print("alphapose available:", alphapose_ready)
if alphapose_error:
    print("Reason:", alphapose_error)


In [ ]:
import urllib.request

import cv2
import matplotlib.pyplot as plt

from posebench.benchmark import write_json
from posebench.export import export_frames_to_csv, export_frames_to_json
from posebench.keypoints_schema import map_tool_keypoints_to_canonical
from posebench.viz import draw_skeleton

sample_url = "https://upload.wikimedia.org/wikipedia/commons/9/92/Walking_person.jpg"
sample_path = repo_root / "assets" / "sample_input_walking.jpg"
if not sample_path.exists():
    urllib.request.urlretrieve(sample_url, sample_path)

bgr = cv2.imread(str(sample_path))
if bgr is None:
    raise RuntimeError("Could not load sample image")

if alphapose_ready:
    status = {
        "tool": "alphapose",
        "status": "not_measured",
        "notes": "AlphaPose detected. Add project-specific inference command in this cell.",
    }
    points = []
else:
    h, w = bgr.shape[:2]
    points = [
        {"x": (0.25 + 0.02 * i) * w, "y": (0.2 + 0.014 * i) * h, "confidence": 0.7}
        for i in range(17)
    ]
    status = {
        "tool": "alphapose",
        "status": "not_measured",
        "notes": "Used synthetic COCO-17 fallback sample because AlphaPose is unavailable.",
    }

canonical = map_tool_keypoints_to_canonical("alphapose", points, min_confidence=0.1)
frames = [
    {
        "frame_index": 0,
        "timestamp_ms": 0.0,
        "person_id": 0,
        "tool": "alphapose",
        "schema": "coco17",
        "keypoints": canonical,
    }
]

export_frames_to_csv(frames, repo_root / "results" / "alphapose_demo_keypoints.csv")
export_frames_to_json(frames, repo_root / "results" / "alphapose_demo_keypoints.json")
write_json(status, repo_root / "results" / "benchmark_raw_alphapose_demo.json")

render = draw_skeleton(bgr, canonical, min_confidence=0.1)
out_path = repo_root / "assets" / "generated" / "alphapose_fallback_overlay.jpg"
out_path.parent.mkdir(parents=True, exist_ok=True)
cv2.imwrite(str(out_path), render)

plt.figure(figsize=(7, 4))
plt.imshow(cv2.cvtColor(render, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title("AlphaPose fallback overlay")
plt.show()

status


## Fallback options

- Local GPU environment with pinned AlphaPose dependencies.
- Docker image with known CUDA/PyTorch compatibility.
